In [1]:
# performance testing for metapop1
import pickle
from metapop_value_iteration import _act, build_optimal_controller_fully_observable
from metapop1 import metapop1
import numpy as np
import pandas as pd


def avgperformance(env, config, policy_printout=False):
    policytype = config['policytype']
    num_episodes = config['num_episodes']
    if policytype == 0: # Value iteration
        print( 'policy type: value iteration')
        with open(f'./value_iteration/VI_controller_setting{config["envid"]}.pkl', 'rb') as f:
            ctrl = pickle.load(f)
        policy = ctrl['policy']
        # reinitiate the environment
        settings = ctrl['envinfo']
        print(settings)
        env = metapop1(settings)
    elif policytype == 1: # ppo
        print( 'policy type: ppo')
        policy = 0
    elif policytype == 2: # heuristics
        if config['heuristics'] == 0:
            print( 'policy type: heuristics, no action')
        elif config['heuristics'] == 1:
            print( 'policy type: heuristics, full action')
    rewards = []
    for i in range(num_episodes):
        obs, state = env.reset()
        input = obs
        done = False
        ep_reward = 0
        while not done:
            if policytype == 0:
                action = _act(env, policy, input)
            elif policytype == 1:
                action = 0
            elif policytype == 2: # heuristics
                if config['heuristics'] == 0: # no action
                    action = np.zeros(env.aS_dim + env.aR_dim)
                elif config['heuristics'] == 1: # full action
                    aR = np.zeros(env.aR_dim)
                    rlen = min(env.kR, env.aR_dim)
                    aR[np.random.choice(np.arange(env.aR_dim), size=rlen, replace=False)] = 1
                    aS = np.zeros(env.aS_dim)
                    slen = min(env.kS, env.aS_dim)
                    aS[np.random.choice(np.arange(env.aS_dim), size=slen, replace=False)] = 1
                    action = np.concatenate((aS, aR))
            if policy_printout:
                print(f't: {input[env.oidx["t"][0]]},  X: {input[env.oidx["X"]]}, aS: {action[env.aidx["aS"]]}, H: {input[env.oidx["H"]]}, Z: {input[env.oidx["Z"]]}, aR: {action[env.aidx["aR"]]}')
            obs, reward, done, info = env.step(action)
            input = env.obs
            ep_reward += reward
        rewards.append(ep_reward)
        if (i+1) % 100 == 0:
            print(f'Episode {i+1} done')
    print(f'Average reward over {num_episodes} episodes: {np.mean(rewards)}')
    return rewards


# policytype: 0 for value iteration, 1 for ppo, 2 for heuristics
config = {'policytype': 0, 'envid': 1, 'POMDP': 0, 'heuristics': 0, 'num_episodes': 1000}
settings = dict(
    partial_observability=0,
    patchnum=4,                 # keep small for exact DP
    action_portfolio=0,
    survey_is_action=0,
    dispersal_regime=1,
    terminal_penalty=0,
    limit_actions=1,
    dispersal_type='uniform',
    dispersal_ID=0,
    paramsetID=1,
    kR=1,
    kS=4
)
env = metapop1(settings)

rewards = avgperformance(env,config=config, policy_printout=False)

policy type: value iteration
{'ID': 1, 'partial_observability': 0, 'patchnum': 4, 'action_portfolio': 0, 'survey_is_action': 0, 'dispersal_regime': 1, 'terminal_penalty': 0, 'limit_actions': 1, 'dispersal_type': 'uniform', 'dispersal_ID': 0, 'paramsetID': 1, 'kR': 1, 'kS': 4}
Episode 100 done
Episode 200 done
Episode 300 done
Episode 400 done
Episode 500 done
Episode 600 done
Episode 700 done
Episode 800 done
Episode 900 done
Episode 1000 done
Average reward over 1000 episodes: 22.684
